In [1]:
!pip install pandas numpy xml

ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.26.0 Requires-Python >=3.9,<3.13; 1.26.1 Requires-Python >=3.9,<3.13
ERROR: Could not find a version that satisfies the requirement xml (from versions: none)
ERROR: No matching distribution found for xml


In [19]:
import os
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET

# =====================================================
# Convert *any* XML table into a tidy DataFrame
# =====================================================
def parse_generic_table(root, table_name):
    rows = []
    table = root.find(table_name)
    if table is None:
        return pd.DataFrame()  # empty
    
    for event in table.findall("event"):
        row = {"datetime": event.get("ts")}
        # extract all attributes automatically
        for attr, value in event.attrib.items():
            if attr == "ts":
                continue
            try:
                row[attr] = float(value)
            except:
                row[attr] = value
        rows.append(row)

    if not rows:
        return pd.DataFrame()

    # Convert datetime
    df = pd.DataFrame(rows)
    df["datetime"] = pd.to_datetime(df["datetime"], format="%d-%m-%Y %H:%M:%S", errors="coerce")
    df = df.dropna(subset=["datetime"]).sort_values("datetime")
    return df.reset_index(drop=True)

# =====================================================
# Load one patient (ALL tables)
# =====================================================
def load_patient(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()

    # list of all sections you want
    sections = [
        "glucose_level", "finger_stick", "basal", "temp_basal", "bolus", "meal",
        "sleep", "work", "stressors", "hypo_event", "illness", "exercise",
        "basis_heart_rate", "basis_gsr", "basis_skin_temperature",
        "basis_air_temperature", "basis_steps", "basis_sleep"
    ]

    dfs = []
    for sec in sections:
        df = parse_generic_table(root, sec)
        if len(df) > 0:
            # rename columns to include table name prefix
            df = df.rename(columns={c: f"{sec}_{c}" for c in df.columns if c != "datetime"})
            dfs.append(df)

    # start with glucose (must exist)
    main = dfs[0].copy()

    # merge everything with tolerance
    for extra in dfs[1:]:
        main = pd.merge_asof(
            main.sort_values("datetime"),
            extra.sort_values("datetime"),
            on="datetime",
            direction="nearest",
            tolerance=pd.Timedelta("3min")
        )

    return main.sort_values("datetime").reset_index(drop=True)

# =====================================================
# Load all patients
# =====================================================
def load_all_patients(data_dir):
    patients = {}
    for f in os.listdir(data_dir):
        if f.endswith(".xml"):
            pid = f.replace(".xml", "")
            patients[pid] = load_patient(os.path.join(data_dir, f))
    return patients

# =====================================================
# Resample + interpolate glucose, forward fill others
# =====================================================
def resample_5min(df):
    df = df.set_index("datetime")

    # keep only numeric columns when resampling
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    df_num = df[numeric_cols].resample("5min").mean()

    # interpolate glucose only
    if "glucose_level_value" in df_num.columns:
        df_num["glucose_level_value"] = df_num["glucose_level_value"].interpolate(limit=3)

    # restore datetime index
    df_num = df_num.reset_index()

    return df_num

# =====================================================
# Add time features
# =====================================================
def add_time_features(df):
    df["hour"] = df["datetime"].dt.hour
    df["minute"] = df["datetime"].dt.minute
    df["dow"] = df["datetime"].dt.dayofweek
    df["day"] = df["datetime"].dt.day
    return df

# =====================================================
# Create supervised sequences
# =====================================================
def create_sequences(df, seq_len=12, pred_horizon=6):
    feature_cols = [c for c in df.columns if c != "datetime"]
    X, y = [], []
    data = df[feature_cols].values
    target = df["glucose_level_value"].values

    for i in range(len(df) - seq_len - pred_horizon):
        X.append(data[i:i+seq_len])
        y.append(target[i+seq_len+pred_horizon])

    return np.array(X), np.array(y)

# =====================================================
# Full pipeline
# =====================================================
def build_ohio_dataset(data_dir, seq_len=12, pred_horizon=6):
    raw = load_all_patients(data_dir)
    X_all, y_all = {}, {}

    for pid, df in raw.items():
        df = resample_5min(df)
        df = add_time_features(df)
        X, y = create_sequences(df, seq_len, pred_horizon)
        X_all[pid] = X
        y_all[pid] = y

    return X_all, y_all

In [20]:
# ==========================================================================
# USAGE
# ==========================================================================
X_all, y_all = build_ohio_dataset("/Users/akki/Desktop/AKKI/college/AI in Healthcare/Project/Blood Glucose Prediction/OhioT1DM model/OhioT1DM", seq_len=12, pred_horizon=6)
X_all["559-ws-training"].shape, y_all["559-ws-training"].shape

KeyError: '559-ws-training'

In [28]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using:", device)
# ============================================================
# 1. CLEAN + NORMALIZE X, y  (fixes ALL NaN and exploding-loss issues)
# ============================================================
def clean_normalize(X, y):
    # remove constant columns
    std = X.std(axis=(0,1))
    keep = std > 1e-6
    X = X[:, :, keep]

    # replace NaN / inf
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

    # normalize X
    mean = X.mean(axis=(0,1), keepdims=True)
    std = X.std(axis=(0,1), keepdims=True) + 1e-6
    X = (X - mean) / std

    # normalize y
    y_mean = y.mean()
    y_std = y.std() + 1e-6
    y = (y - y_mean) / y_std

    return X.astype(np.float32), y.astype(np.float32), keep, (mean, std, y_mean, y_std)


# ============================================================
# 2. Dataset Wrapper
# ============================================================
class TimeXDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# ============================================================
# 3. TimeX Transformer Model
# ============================================================
class TimeX(nn.Module):
    def __init__(self, input_dim, seq_len, d_model=128, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.fc = nn.Linear(d_model * seq_len, 1)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.encoder(x)
        x = x.reshape(x.size(0), -1)
        return self.fc(x).squeeze()


# ============================================================
# 4. Train TimeX Model (Stable + Clipping)
# ============================================================
def train_timex(X, y, seq_len, epochs=50, batch=32, lr=1e-4):

    # ----- clean & normalize -----
    X, y, keep, norms = clean_normalize(X, y)

    dataset = TimeXDataset(X, y)
    n = len(dataset)

    train_size = int(0.8 * n)
    val_size = n - train_size
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=batch, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch)

    model = TimeX(input_dim=X.shape[2], seq_len=seq_len).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for Xb, yb in train_loader:
            optimizer.zero_grad()
            Xb = Xb.to(device)
            yb = yb.to(device)

            pred = model(Xb)
            loss = criterion(pred, yb)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb = Xb.to(device)
                yb = yb.to(device)
                pred = model(Xb)
                val_loss += criterion(pred, yb).item()

        print(f"Epoch {epoch+1}/{epochs} | Train Loss={train_loss/len(train_loader):.4f} | Val Loss={val_loss/len(val_loader):.4f}")

    return model, keep, norms


Using: mps


In [29]:
X_all, y_all = build_ohio_dataset("/Users/akki/Desktop/AKKI/college/AI in Healthcare/Project/Blood Glucose Prediction/OhioT1DM model/OhioT1DM", seq_len=12, pred_horizon=6)

# pid = "Patient_1"
# X = X_all[pid]
# y = y_all[pid]

# model, keep, norms = train_timex(X, y, seq_len=12, epochs=50)

In [33]:
# ONE BLOCK: train/eval/save/plot for all patients
import os
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
from math import sqrt

# --- adjust paths ---
OUT_DIR = "timex_results"
os.makedirs(OUT_DIR, exist_ok=True)
MODELS_DIR = os.path.join(OUT_DIR, "models")
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# --- reuse train_timex(X,y,seq_len,epochs,...) function from prior block ---
# It must return: model, keep, norms
# norms = (mean, std, y_mean, y_std)
# keep is boolean vector of kept channels (applied before normalization)

# If you don't have the function in scope, import or paste it here.
# ---------------------------------------------------------------------
# Helper: prepare input X for inference using keep & norms
def prepare_for_inference(X_raw, keep, norms):
    """
    X_raw: (N, seq_len, C_raw) original array from build_ohio_dataset
    keep: boolean mask of length C_raw that was used/trained
    norms: (mean, std, y_mean, y_std) from training; mean/std shapes (1,1,C_kept)
    """
    # select kept channels
    X = X_raw[:, :, keep]
    mean, std, y_mean, y_std = norms
    # normalize with training mean/std
    X = (X - mean) / (std + 1e-12)
    return X.astype(np.float32)

# ---------------------------------------------------------------------
# Helper: denormalize predictions & compute metrics
def denormalize_and_metrics(y_pred_norm, y_true_norm, norms):
    mean_x, std_x, y_mean, y_std = norms
    # denormalize y
    y_pred = y_pred_norm * y_std + y_mean
    y_true = y_true_norm * y_std + y_mean
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    mae = float(np.mean(np.abs(y_pred - y_true)))
    return y_pred, y_true, rmse, mae

# ---------------------------------------------------------------------
# Helper: run model to predict on full X (returns normalized preds)
def predict_model(model, X_norm):
    model.eval()
    device = next(model.parameters()).device
    Xt = torch.tensor(X_norm, dtype=torch.float32).to(device)
    with torch.no_grad():
        preds = model(Xt).cpu().numpy()
    return preds

# ---------------------------------------------------------------------
# Main orchestration: train per-patient, evaluate, save model/plot
def run_all_patients(X_all, y_all, seq_len=12, epochs=50, batch=32, lr=1e-4, device=None):
    if device is None:
        device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


    results = []
    for pid, X in X_all.items():
        try:
            y = y_all[pid]
            if len(X) < 50:
                print(f"Skipping {pid} (too few sequences: {len(X)})")
                continue

            print(f"\n=== Patient {pid}: sequences={len(X)}, channels={X.shape[2]} ===")

            # train_timex is expected to handle cleaning/normalization internally and return (model, keep, norms)
            # if train_timex is in a module, import it; otherwise ensure it's in scope.
            model, keep, norms = train_timex(X, y, seq_len=seq_len, epochs=epochs, batch=batch, lr=lr)

            # move model to device & save best by final val loss (train_timex printed logs only)
            model.to(device)

            # Save model weights
            model_path = os.path.join(MODELS_DIR, f"timex_{pid}.pt")
            torch.save({
                "model_state_dict": model.state_dict(),
                "keep": keep,
                "norms": norms,
                "seq_len": seq_len
            }, model_path)

            # Prepare X for inference (must apply same keep & norms)
            X_norm = prepare_for_inference(X, keep, norms)

            # Predict (normalized output)
            y_pred_norm = predict_model(model, X_norm)
            y_pred_norm = np.nan_to_num(y_pred_norm, nan=0.0, posinf=1e6, neginf=-1e6)

            # Ensure shapes align — model predicts one value per sequence; construct y_true_norm aligned
            # The training code used X windows and predicted y at index seq_len+pred_horizon.
            # Our X_all and y_all are already sequences and targets aligned; y is target array used in training (normalized inside train_timex).
            # But train_timex returned norms including y_mean/y_std that were applied to y internally.
            # We need the normalized y values used during training to compute metrics; re-normalize original y using norms:
            _, _, y_mean, y_std = norms
            y_norm = (y - y_mean) / (y_std + 1e-12)
            y_norm = np.nan_to_num(y_norm, nan=0.0, posinf=0.0, neginf=0.0)

            # If lengths mismatch (e.g., due to random_split ordering) truncate to min length
            n_eval = min(len(y_pred_norm), len(y_norm))
            y_pred_norm = y_pred_norm[:n_eval]
            y_norm = y_norm[:n_eval]

            # Denormalize and compute real mg/dL RMSE & MAE
            y_pred, y_true, rmse, mae = denormalize_and_metrics(y_pred_norm, y_norm, norms)

            # Save small evaluation plot (first 500 points or full if smaller)
            nplot = min(500, n_eval)
            plt.figure(figsize=(10,4))
            plt.plot(y_true[:nplot], label="True (mg/dL)")
            plt.plot(y_pred[:nplot], label="Pred (mg/dL)", alpha=0.8)
            plt.title(f"Patient {pid} — RMSE={rmse:.2f} MAE={mae:.2f}")
            plt.xlabel("time-step")
            plt.ylabel("BG (mg/dL)")
            plt.legend()
            plt.tight_layout()
            plot_path = os.path.join(PLOTS_DIR, f"pred_{pid}.png")
            plt.savefig(plot_path)
            plt.close()

            # Append results
            results.append({
                "patient_id": pid,
                "n_sequences": len(X),
                "rmse_mgdL": rmse,
                "mae_mgdL": mae,
                "model_path": model_path,
                "plot_path": plot_path
            })

            print(f"Patient {pid}: RMSE={rmse:.2f} MAE={mae:.2f} | model saved -> {model_path}")

        except Exception as e:
            print(f"ERROR for patient {pid}: {e}")

    # final table
    df_res = pd.DataFrame(results).sort_values("patient_id")
    csv_path = os.path.join(OUT_DIR, "evaluation_table.csv")
    df_res.to_csv(csv_path, index=False)
    print(f"\nSaved evaluation table -> {csv_path}")
    return df_res

# ---------------------------------------------------------------------
# Example usage (uncomment and run in your environment where X_all,y_all exist):
df_results = run_all_patients(X_all, y_all, seq_len=12, epochs=80, batch=32, lr=1e-4)
print(df_results.head())


=== Patient 575-ws-training: sequences=13086, channels=15 ===
Epoch 1/80 | Train Loss=0.9868 | Val Loss=0.9363
Epoch 2/80 | Train Loss=0.9242 | Val Loss=0.8775
Epoch 3/80 | Train Loss=0.8804 | Val Loss=0.8390
Epoch 4/80 | Train Loss=0.8555 | Val Loss=0.8234
Epoch 5/80 | Train Loss=0.8220 | Val Loss=0.8039
Epoch 6/80 | Train Loss=0.8001 | Val Loss=0.7664
Epoch 7/80 | Train Loss=0.7962 | Val Loss=0.8102
Epoch 8/80 | Train Loss=0.7763 | Val Loss=0.7408
Epoch 9/80 | Train Loss=0.7566 | Val Loss=0.7259
Epoch 10/80 | Train Loss=0.7375 | Val Loss=0.7107
Epoch 11/80 | Train Loss=0.7328 | Val Loss=0.7350
Epoch 12/80 | Train Loss=0.7197 | Val Loss=0.7712
Epoch 13/80 | Train Loss=0.6985 | Val Loss=0.8152
Epoch 14/80 | Train Loss=0.6678 | Val Loss=0.6989
Epoch 15/80 | Train Loss=0.6378 | Val Loss=0.7139
Epoch 16/80 | Train Loss=0.6220 | Val Loss=0.5659
Epoch 17/80 | Train Loss=0.6051 | Val Loss=0.5917
Epoch 18/80 | Train Loss=0.5678 | Val Loss=0.5483
Epoch 19/80 | Train Loss=0.5624 | Val Loss=0.5

KeyboardInterrupt: 

In [34]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt


# ============================================================
# DEVICE (MPS FIRST)
# ============================================================
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)


# ============================================================
# SAFE NORMALIZATION FOR OHIO T1DM
# ============================================================
def clean_normalize(X, y):
    # Drop constant channels
    std = X.std(axis=(0,1))
    keep = std > 1e-6
    X = X[:, :, keep]

    # Replace nan/infs
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

    # Normalize X
    mean = X.mean(axis=(0,1), keepdims=True)
    std = X.std(axis=(0,1), keepdims=True) + 1e-6
    X = (X - mean) / std

    # Normalize y
    y_mean = y.mean()
    y_std = y.std() + 1e-6
    y = (y - y_mean) / y_std

    return X.astype(np.float32), y.astype(np.float32), keep, (mean, std, y_mean, y_std)


# ============================================================
# DATASET
# ============================================================
class TimeXDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# ============================================================
# TIMEX TRANSFORMER (DEPTH = 6)
# ============================================================
class TimeX(nn.Module):
    def __init__(self, input_dim, seq_len, d_model=128, n_heads=4, n_layers=6, dropout=0.1):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            batch_first=True,
            dropout=dropout
        )

        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)

        self.fc = nn.Linear(d_model * seq_len, 1)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.encoder(x)
        x = x.reshape(x.size(0), -1)
        return self.fc(x).squeeze()


# ============================================================
# TRAINING FUNCTION (SCHEDULER + EARLY STOP)
# ============================================================
def train_timex(
    X_raw, y_raw,
    seq_len,
    epochs=120,
    batch=32,
    lr=1e-4,
    patience=15
):

    X, y, keep, norms = clean_normalize(X_raw, y_raw)

    dataset = TimeXDataset(X, y)
    n = len(dataset)
    train_size = int(0.8 * n)
    val_size = n - train_size

    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])
    train_loader = DataLoader(train_ds, batch_size=batch, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch, shuffle=False)

    model = TimeX(input_dim=X.shape[2], seq_len=seq_len).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    criterion = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    wait = 0

    for epoch in range(1, epochs+1):

        model.train()
        train_loss = 0

        for Xb, yb in train_loader:
            Xb = Xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(Xb)
            loss = criterion(pred, yb)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb = Xb.to(device)
                yb = yb.to(device)
                pred = model(Xb)
                val_loss += criterion(pred, yb).item()

        train_loss /= len(train_loader)
        val_loss /= len(val_loader)

        scheduler.step()

        print(f"Epoch {epoch}/{epochs} | Train={train_loss:.4f} | Val={val_loss:.4f}")

        # Early stopping
        if val_loss < best_val:
            best_val = val_loss
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)
    return model, keep, norms, best_val


# ============================================================
# PREDICTION UTILS
# ============================================================
def prepare_for_inference(X_raw, keep, norms):
    X = X_raw[:, :, keep]
    mean, std, y_mean, y_std = norms
    X = (X - mean) / (std + 1e-12)
    return X.astype(np.float32)


def denormalize_and_metrics(y_pred_norm, y_true_norm, norms):
    mean_x, std_x, y_mean, y_std = norms
    y_pred = y_pred_norm * y_std + y_mean
    y_true = y_true_norm * y_std + y_mean

    mask = ~np.isnan(y_pred) & ~np.isnan(y_true)
    y_pred = y_pred[mask]
    y_true = y_true[mask]

    if len(y_pred) == 0:
        return float("nan"), float("nan")

    rmse = float(np.sqrt(np.mean((y_pred - y_true)**2)))
    mae = float(np.mean(np.abs(y_pred - y_true)))
    return rmse, mae


def predict_model(model, X_norm):
    model.eval()
    Xt = torch.tensor(X_norm, dtype=torch.float32).to(device)
    with torch.no_grad():
        preds = model(Xt).cpu().numpy()
    return preds


# ============================================================
# FULL PER-PATIENT TRAINING LOOP
# ============================================================
def train_all_patients(X_all, y_all, seq_len=12, epochs=120, lr=1e-4):

    os.makedirs("timex_results/models", exist_ok=True)
    os.makedirs("timex_results/plots", exist_ok=True)

    results = []

    for pid in X_all:
        X_raw = X_all[pid]
        y_raw = y_all[pid]

        if len(X_raw) < 50:
            print(f"Skipping {pid}, too few sequences.")
            continue

        print(f"\n=== PATIENT {pid} ===")

        model, keep, norms, best_val = train_timex(
            X_raw, y_raw, seq_len,
            epochs=epochs, lr=lr
        )

        X_norm = prepare_for_inference(X_raw, keep, norms)
        y_pred_norm = predict_model(model, X_norm)

        _, _, y_mean, y_std = norms
        y_norm = (y_raw - y_mean) / (y_std + 1e-12)

        n = min(len(y_pred_norm), len(y_norm))
        y_pred_norm = y_pred_norm[:n]
        y_norm = y_norm[:n]

        rmse, mae = denormalize_and_metrics(y_pred_norm, y_norm, norms)
        print(f"Patient {pid}: RMSE={rmse:.2f} MAE={mae:.2f}")

        # Save model
        torch.save({
            "state_dict": model.state_dict(),
            "keep": keep,
            "norms": norms
        }, f"timex_results/models/timex_{pid}.pt")

        # Plot
        plt.figure(figsize=(10,4))
        plt.plot(y_norm * y_std + y_mean, label="True", alpha=0.8)
        plt.plot(y_pred_norm * y_std + y_mean, label="Pred", alpha=0.8)
        plt.title(f"{pid} — RMSE={rmse:.2f} MAE={mae:.2f}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"timex_results/plots/{pid}.png")
        plt.close()

        results.append([pid, rmse, mae, best_val])

    df = pd.DataFrame(results, columns=["patient", "rmse", "mae", "best_val"])
    df.to_csv("timex_results/evaluation.csv", index=False)
    return df


# ============================================================
# POOLED MODEL TRAINING
# ============================================================
def train_pooled_model(X_all, y_all, seq_len=12, epochs=120, lr=1e-4):
    X_pool = np.concatenate([X_all[p] for p in X_all], axis=0)
    y_pool = np.concatenate([y_all[p] for p in y_all], axis=0)
    model, keep, norms, best_val = train_timex(X_pool, y_pool, seq_len, epochs=epochs, lr=lr)
    torch.save({
        "state_dict": model.state_dict(),
        "keep": keep,
        "norms": norms
    }, "timex_results/models/timex_pooled.pt")
    return model, keep, norms

Device: mps


In [37]:

X_all, y_all = build_ohio_dataset("/Users/akki/Desktop/AKKI/college/AI in Healthcare/Project/Blood Glucose Prediction/OhioT1DM model/OhioT1DM", seq_len=12, pred_horizon=6)


In [39]:
df_results = train_all_patients(X_all, y_all, seq_len=12, epochs=120, lr=1e-4)
df_results


=== PATIENT 575-ws-training ===
Epoch 1/120 | Train=0.9730 | Val=0.9845
Epoch 2/120 | Train=0.9072 | Val=0.8969
Epoch 3/120 | Train=0.8797 | Val=0.9086
Epoch 4/120 | Train=0.8360 | Val=0.8290
Epoch 5/120 | Train=0.8177 | Val=0.8037
Epoch 6/120 | Train=0.7938 | Val=0.8016
Epoch 7/120 | Train=0.7489 | Val=0.7544
Epoch 8/120 | Train=0.7477 | Val=0.7421
Epoch 9/120 | Train=0.7345 | Val=0.6814
Epoch 10/120 | Train=0.6965 | Val=0.6395
Epoch 11/120 | Train=0.6902 | Val=0.6300
Epoch 12/120 | Train=0.6613 | Val=0.6412
Epoch 13/120 | Train=0.6502 | Val=0.7248
Epoch 14/120 | Train=0.6622 | Val=0.5969
Epoch 15/120 | Train=0.6217 | Val=0.5749
Epoch 16/120 | Train=0.5994 | Val=0.5540
Epoch 17/120 | Train=0.5884 | Val=0.6324
Epoch 18/120 | Train=0.5897 | Val=0.5744
Epoch 19/120 | Train=0.5752 | Val=0.5415
Epoch 20/120 | Train=0.5616 | Val=0.5755
Epoch 21/120 | Train=0.5506 | Val=0.4958
Epoch 22/120 | Train=0.5544 | Val=0.5681
Epoch 23/120 | Train=0.5313 | Val=0.5092
Epoch 24/120 | Train=0.5107 | Val

,patient,rmse,mae,best_val
0,575-ws-training,17.865262,12.368915,0.131495
1,563-ws-training,10.859438,7.610897,0.044666
2,559-ws-testing,14.799761,10.118010,0.070698
3,588-ws-testing,11.437453,8.517635,0.052411
4,Patient_1,17.487444,11.494884,0.065873
5,570-ws-testing,14.025631,9.654736,0.066331
6,588-ws-training,12.569339,8.863908,0.054120
7,563-ws-testing,12.019305,8.670820,0.052588
8,570-ws-training,11.270082,7.082795,0.050937
9,591-ws-training,14.592848,9.892788,0.046584


# Presentation

https://docs.google.com/presentation/d/e/2PACX-1vQsA122Li0C2cOY_459MWIO_yF8pECi_TQh48wZqToNWjJvMa9SGF3KVCU_BtP4GVfSZk5LF63L4kCi/pub?start=false&loop=false&delayms=3000